# 02 — Gather DeFiLlama data

## What this notebook does

This notebook collects the actual data used in the project.

We use DeFiLlama to get:
- **historical TVL** for the selected protocols
- **current APY** where DeFiLlama provides a yield page

The result will be two datasets:

1. **`staking_tvl_timeseries.csv`**  
   Main analysis dataset with one row per date and protocol

2. **`staking_protocol_summary.csv`**  
   Small summary table with current TVL and APY (if available)

Important:
- This notebook is **data-first**
- We keep metadata to a minimum
- Native staking is **not included as a DeFiLlama protocol row**, because it is a baseline mechanism rather than a normal DeFi protocol entry

In [1]:
import re
import requests
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

## Protocols used in the data collection

These are the DeFiLlama protocol slugs we use:
- Lido => `lido`
- Rocket Pool => `rocket-pool`
- EigenLayer => `eigencloud` in DeFiLlama

In [2]:
# Here we add the protocols that we want to compare
# We also analyze native staking, but we dont add it here, because we manually add it later
protocols = [
    {"protocol": "Lido", "category": "liquid_staking", "slug": "lido", "has_yield_page": True},
    {"protocol": "Rocket Pool", "category": "liquid_staking", "slug": "rocket-pool", "has_yield_page": True},
    {"protocol": "EigenLayer", "category": "restaking", "slug": "eigencloud", "has_yield_page": False},
]

protocols_df = pd.DataFrame(protocols)
protocols_df

,protocol,category,slug,has_yield_page
0,Lido,liquid_staking,lido,True
1,Rocket Pool,liquid_staking,rocket-pool,True
2,EigenLayer,restaking,eigencloud,False


## Helper functions

In [3]:
# Get the current TVL for one protocol from DeFiLlama
def fetch_current_tvl(slug: str):
    url = f"https://api.llama.fi/tvl/{slug}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    value = r.json()
    if isinstance(value, (int, float)):
        return float(value), url
    return pd.NA, url

# Get the full protocol. Use to extract TVL history
def fetch_protocol_json(slug: str):
    url = f"https://api.llama.fi/protocol/{slug}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json(), url

# DeFiLlama can name the TVl column differently.
# Here we standardize it to date + tvl_usd
def normalize_tvl_df(df: pd.DataFrame):
    # Check for common tvl names and take the first one if exists
    possible_value_cols = ["totalLiquidityUSD", "liquidity", "totalLiquidityUsd", "tvl", "value"]
    value_col = next((c for c in possible_value_cols if c in df.columns), None)
    
    # If we dont have both a date and a TVL we return an empty df
    if "date" not in df.columns or value_col is None:
        return pd.DataFrame(columns=["date", "tvl_usd"])
    
    # Keep only the date column and the found TVL
    out = df[["date", value_col]].copy()
    # Rename for consistency
    out = out.rename(columns={value_col: "tvl_usd"})
    # Convert UNIX timetsamp to date
    out["date"] = pd.to_datetime(out["date"], unit="s", errors="coerce")
    out["tvl_usd"] = pd.to_numeric(out["tvl_usd"], errors="coerce")
    out = out.dropna(subset=["date", "tvl_usd"]).sort_values("date").reset_index(drop=True)
    return out

# Try to extract a usable TVL history
# First try the overall TVL history and if it doesnt work,
# we try chain-level histories with Ethereum as a preference
def extract_tvl_timeseries(protocol_json: dict):
    # Some protocols have a direct overall TVL history under "tvl"
    tvl_obj = protocol_json.get("tvl")
    if isinstance(tvl_obj, list) and len(tvl_obj) > 0:
        return normalize_tvl_df(pd.DataFrame(tvl_obj))
    
    # If no direct history, look into chain-level history
    chain_tvls = protocol_json.get("chainTvls", {})
    if isinstance(chain_tvls, dict) and len(chain_tvls) > 0:
        # Start with all available chains
        preferred_keys = list(chain_tvls.keys())
        # If Ethereum is present, we take that 
        if "Ethereum" in chain_tvls:
            preferred_keys = ["Ethereum"] + [k for k in preferred_keys if k != "Ethereum"]
        
        # Go through the selected chains
        for key in preferred_keys:
            entry = chain_tvls.get(key)
            # Check if chain is a dictionary
            if isinstance(entry, dict):
                # Different protocols may store the TVL history udner different nested keys
                for nested_key in ["tvl", "tokensInUsd", "tokens"]:
                    nested = entry.get(nested_key)
                    # If we find a non-empty list, normalize it
                    if isinstance(nested, list) and len(nested) > 0:
                        return normalize_tvl_df(pd.DataFrame(nested))

    return pd.DataFrame(columns=["date", "tvl_usd"])

# For protocols with a DeFiLlama yield page, try to get the displayed average APY
def fetch_average_apy(slug: str):
    url = f"https://defillama.com/protocol/yields/{slug}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        text = r.text
        # Look for a phrase like "average APY 2.43%"
        match = re.search(r"average APY\s*([0-9]+(?:\.[0-9]+)?)%", text, flags=re.IGNORECASE)
        # If the pattern is found
        if match:
            # return the APY, url, and rate type
            return float(match.group(1)), url, "APY"
        return pd.NA, url, "APY not found on yield page"
    except Exception:
        return pd.NA, url

## Collect the data

In [4]:
timeseries_frames = []
summary_rows = []

for row in protocols:
    protocol = row["protocol"]
    category = row["category"]
    slug = row["slug"]
    has_yield_page = row["has_yield_page"]
    
    # Current TVL gives us a quick size comparison
    current_tvl, tvl_source = fetch_current_tvl(slug)
    # The protocol KSON is used to recover TVL history
    protocol_json, history_source = fetch_protocol_json(slug)
    hist_df = extract_tvl_timeseries(protocol_json)
    
    # Only get APY where a DeFiLlama yield page exists
    if has_yield_page:
        annual_rate_percent, rate_source, rate_type = fetch_average_apy(slug)
    else:
        annual_rate_percent = pd.NA
        rate_source = "No DeFiLlama yield page used"
        rate_type = pd.NA

    # Add protocol label columns
    if not hist_df.empty:
        hist_df["protocol"] = protocol
        hist_df["category"] = category
        hist_df["slug"] = slug
        timeseries_frames.append(hist_df)
    
    summary_rows.append({
        "protocol": protocol,
        "category": category,
        "slug": slug,
        "current_tvl_usd": current_tvl,
        "annual_rate_percent": annual_rate_percent,
        "rate_type": rate_type,
        "source": rate_source,
        "note": pd.NA,
        "tvl_source": tvl_source,
        "history_source": history_source
    })

staking_tvl_timeseries_df = pd.concat(timeseries_frames, ignore_index=True)
staking_protocol_summary_df = pd.DataFrame(summary_rows)

staking_tvl_timeseries_df.head()

,date,tvl_usd,protocol,category,slug
0,2020-12-19 23:00:00,1484680,Lido,liquid_staking,lido
1,2020-12-20 23:00:00,2697598,Lido,liquid_staking,lido
2,2020-12-21 23:00:00,3410253,Lido,liquid_staking,lido
3,2020-12-22 23:00:00,4563625,Lido,liquid_staking,lido
4,2020-12-23 23:00:00,4661610,Lido,liquid_staking,lido


In [5]:
# Native staking we can use for a benchmark
# We will add it manually to the table with information from ethereum.org
summary_rows.append({
    "protocol": "Ethereum Native Staking",
    "category": "native_staking",
    "slug": pd.NA,
    "current_tvl_usd": pd.NA,
    "annual_rate_percent": 2.8,
    "rate_type": "APR",
    "source": "https://ethereum.org/staking/",
    "note": "Benchmark row from ethereum.org staking page; Total ETH staked: 38,746,608; Validators: 899,017",
    "tvl_source": pd.NA,
    "history_source": pd.NA
})

# Rebuild the time series
staking_tvl_timeseries_df = (
    pd.concat(timeseries_frames, ignore_index=True)
    if timeseries_frames else
    pd.DataFrame(columns=["date", "tvl_usd", "protocol", "category", "slug"])
)

staking_protocol_summary_df = pd.DataFrame(summary_rows)

# Clean the time series
if not staking_tvl_timeseries_df.empty:
    staking_tvl_timeseries_df["tvl_usd"] = pd.to_numeric(
        staking_tvl_timeseries_df["tvl_usd"], errors="coerce"
    )
    staking_tvl_timeseries_df = staking_tvl_timeseries_df.dropna(subset=["tvl_usd"])
    staking_tvl_timeseries_df = staking_tvl_timeseries_df.sort_values(
        ["protocol", "date"]
    ).reset_index(drop=True)

# Make sure TVL is numeric
staking_protocol_summary_df["current_tvl_usd"] = pd.to_numeric(
    staking_protocol_summary_df["current_tvl_usd"], errors="coerce"
)

## Save the datasets

In [6]:
timeseries_path = DATA_PROCESSED / "staking_tvl_timeseries.csv"
summary_path = DATA_PROCESSED / "staking_protocol_summary.csv"

staking_tvl_timeseries_df.to_csv(timeseries_path, index=False)
staking_protocol_summary_df.to_csv(summary_path, index=False)

In [9]:
staking_tvl_timeseries_df

,date,tvl_usd,protocol,category,slug
0,2023-06-14 00:00:00,13300350,EigenLayer,restaking,eigencloud
1,2023-06-15 00:00:00,16484319,EigenLayer,restaking,eigencloud
2,2023-06-16 00:00:00,16557169,EigenLayer,restaking,eigencloud
3,2023-06-17 00:00:00,17099837,EigenLayer,restaking,eigencloud
4,2023-06-18 00:00:00,17177115,EigenLayer,restaking,eigencloud
...,...,...,...,...,...
4731,2026-05-14 00:00:00,1130286171,Rocket Pool,liquid_staking,rocket-pool
4732,2026-05-15 00:00:00,1137796615,Rocket Pool,liquid_staking,rocket-pool
4733,2026-05-16 00:00:00,1106988804,Rocket Pool,liquid_staking,rocket-pool
4734,2026-05-17 00:00:00,1083476389,Rocket Pool,liquid_staking,rocket-pool


## What we have now

After this notebook, we have:
- a **time-series dataset** for TVL
- a **small summary table** for current TVL and APY

This is enough to move to analysis.

The next notebook should answer questions like:
- How did TVL evolve over time for the selected protocols?
- How large is restaking compared with liquid staking?
- What APY values are available directly from DeFiLlama, and where are they missing?